# Baseline path following

This notebook implements the baseline computer vision by applying a HSV mask and dividing sections of the screen to determine what direction should be inputted at that moment

In [ ]:
import ipywidgets.widgets as widgets
from IPython.display import display, HTML
import sys
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import traitlets
from traitlets.config.configurable import SingletonConfigurable
import time

# Import your motor control class and initialise
import motors
robot = motors.MotorsYukon(mecanum=False)

# Global variable to remember the last turn direction
last_turn_action = None
turn_lock_timeout = 0.0 

# Create widgets for the displaying of the images
display_color = widgets.Image(format='jpeg', width='33%')
display_processed = widgets.Image(format='jpeg', width='33%') 
layout = widgets.Layout(width='100%')

sidebyside = widgets.HBox([display_color, display_processed], layout=layout) 
display(sidebyside) 

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

# Define Camera class using Traitlets
class Camera(SingletonConfigurable):
    color_value = traitlets.Any() 
    
    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA # VGA (672*376)
        init_params.coordinate_units = sl.UNIT.MILLIMETER  
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open : "+repr(status)+". Exit program.")
            self.zed.close()
            exit(1)

        self.runtime = sl.RuntimeParameters()
        self.thread_running_flag = False
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        
        fps = camera_info.camera_configuration.fps
        fourcc = cv2.VideoWriter_fourcc(*'XVID') 
        self.video_writer = cv2.VideoWriter('color_video_f320951.avi', fourcc, fps, (self.width, self.height))
        print(f"Recording Started: color_video_f320951.avi at {fps} FPS")
        self.image = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        start_time = time.time()
        while(self.thread_running_flag == True):
            # Record for 150 seconds
            if time.time() - start_time >= 150.0:
                print("\n150 seconds reached. Stopping robot and saving AVI video...")
                self.thread_running_flag = False
                break
                
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                color_value_BGRA = self.image.get_data()
                # Convert to standard BGR for OpenCV and your Neural Network
                self.color_value = cv2.cvtColor(color_value_BGRA, cv2.COLOR_BGRA2BGR)
                # Write this specific frame to the .avi file
                self.video_writer.write(self.color_value)
                
            time.sleep(0.001)
            
        self.video_writer.release()
        self.zed.close()
        robot.stop() # 
        print("AVI file saved.")
                
    def start(self): 
        if self.thread_running_flag == False: 
            self.thread_running_flag = True 
            self.thread = threading.Thread(target=self._capture_frames) 
            self.thread.start() 

    def stop(self): 
        if self.thread_running_flag == True:
            self.thread_running_flag = False 
            self.thread.join() 
            self.zed.close()

def process_vision(change):
    global last_turn_action, turn_lock_timeout   

    frame = change['new'].copy()
    
    # Active region is Y: 94 to 282, X: 168 to 504.
    frame[:94, :] = (0, 0, 0)
    frame[282:, :] = (0, 0, 0)
    frame[:, :168] = (0, 0, 0)
    frame[:, 504:] = (0, 0, 0)
    # HSV masking to isolate yellow line
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV) 
    mask = cv2.inRange(hsv, (10, 60, 60), (45, 255, 255))
    #mask = cv2.inRange(hsv, (14, 30, 30), (45, 255, 255)) 

    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel) 
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) 

    # Define zones
    y_end_check = 282
    y_start_check_side = 262   # 20 pixels tall for Left/Right
    y_start_check_center = 222 # 60 pixels tall for Forward box
    
    left_bound = 168
    right_bound = 504
    width = right_bound - left_bound
    # Split the width into three equal segments
    left_zone_end = left_bound + (width // 3)
    right_zone_start = left_bound + (2 * width // 3)
    
    # Isolate the mask arrays for each zone with their specific heights
    bottom_left_zone = mask[y_start_check_side:y_end_check, left_bound:left_zone_end]
    bottom_center_zone = mask[y_start_check_center:y_end_check, left_zone_end:right_zone_start]
    bottom_right_zone = mask[y_start_check_side:y_end_check, right_zone_start:right_bound]
    
    # If there are 20 or more yellow pixels in a zone, trigger  that action
    pixel_threshold = 20 * 255 
    
    touching_left = np.sum(bottom_left_zone) > pixel_threshold
    touching_right = np.sum(bottom_right_zone) > pixel_threshold
    touching_center = np.sum(bottom_center_zone) > pixel_threshold
    
    # Stop switching between left and right turns too rapidly by implementing a cooldown timer
    current_time = time.time()
    # If we are in a cooldown period, ignore the opposite turn trigger
    if current_time < turn_lock_timeout:
        if last_turn_action == 'left':
            touching_right = False
        elif last_turn_action == 'right':
            touching_left = False

    # Motor control logic based on the zones detected
    if touching_left: 
        last_turn_action = 'left' 
        turn_lock_timeout = current_time + 0.5 # Lock out right turns for 0.5 seconds
        robot.left(0.2)
        cv2.putText(frame, "TURN LEFT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
    elif touching_right:
        last_turn_action = 'right' 
        turn_lock_timeout = current_time + 0.5 # Lock out left turns for 0.5 seconds
        robot.right(0.2)
        cv2.putText(frame, "TURN RIGHT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
    elif touching_center:
        robot.forward(0.5)
        cv2.putText(frame, "FORWARD", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
    else:
        # Fallback to the last known turn if the line is lost
        if last_turn_action == 'left':
            robot.left(0.3)
            cv2.putText(frame, "LOST LINE - TURN LEFT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)
        elif last_turn_action == 'right':
            robot.right(0.3)
            cv2.putText(frame, "LOST LINE - TURN RIGHT", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)
        else:
            # If no previous turn is recorded, stop safely
            robot.stop()
            cv2.putText(frame, "STOP - NO LINE DATA", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Draw the 3 zones on the camera feed
    cv2.rectangle(frame, (left_bound, y_start_check_side), (left_zone_end, y_end_check), (255, 0, 0), 2)
    cv2.rectangle(frame, (left_zone_end, y_start_check_center), (right_zone_start, y_end_check), (0, 255, 0), 2)
    cv2.rectangle(frame, (right_zone_start, y_start_check_side), (right_bound, y_end_check), (0, 0, 255), 2)
    
    # Update the display widgets with the resized images
    scale = 0.5 
    resized_color = cv2.resize(frame, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    
    display_color.value = bgr8_to_jpeg(resized_color)
    display_processed.value = bgr8_to_jpeg(resized_mask)

# Create camera object and attach observer
camera = Camera()
camera.observe(process_vision, names=['color_value'])
camera.start()

ModuleNotFoundError: No module named 'pyzed.sl'; 'pyzed' is not a package

In [5]:
import motors

robot = motors.MotorsYukon(mecanum=False)
robot.stop()